In [1]:
import sys, os, glob, tempfile, time
import numpy as np, pandas as pd, SimpleITK as sitk
sys.path.insert(0, "../regcheck")
from regcheck import run_ants, run_unigradicon, brain_points, ncc_after

def random_warp(img, seed, max_rot_deg=8, max_shift_mm=8, max_bend_mm=5, extra_rot_deg=0):
    rng = np.random.default_rng(seed)
    center = img.TransformContinuousIndexToPhysicalPoint([(s - 1) / 2 for s in img.GetSize()])
    rigid = sitk.Euler3DTransform(); rigid.SetCenter(center)
    rot = rng.uniform(-max_rot_deg, max_rot_deg, 3)
    if extra_rot_deg:
        rot[2] += np.sign(rng.uniform(-1, 1)) * extra_rot_deg      # big extra turn around one axis
    rigid.SetRotation(*np.deg2rad(rot)); rigid.SetTranslation(rng.uniform(-max_shift_mm, max_shift_mm, 3))
    bend = sitk.BSplineTransformInitializer(img, [6, 6, 6], order=3)
    p = np.clip(rng.normal(0, max_bend_mm / 2, len(bend.GetParameters())), -max_bend_mm, max_bend_mm)
    bend.SetParameters(p.tolist())
    return sitk.CompositeTransform([rigid, bend])

t1 = sorted(glob.glob("C:/data/brats/**/*-t1n.nii.gz", recursive=True))
africa = [f for f in t1 if "BraTS-SSA" in os.path.basename(f) and "95_Glioma" in f]
gli = [f for f in t1 if "BraTS-GLI" in os.path.basename(f)]
rng = np.random.default_rng(1)                      # seed 1: different cases from part 1
bench = [("Africa", f) for f in rng.choice(africa, 15, replace=False)] + \
        [("GLI", f) for f in rng.choice(gli, 15, replace=False)]
print(len(bench), "bench cases")

out_csv = "../results/bench_regcheck.csv"
done = set(map(tuple, pd.read_csv(out_csv)[["case", "condition"]].values)) if os.path.exists(out_csv) else set()
for i, (group, path) in enumerate(bench):
    case = os.path.basename(path).replace("-t1n.nii.gz", "")
    for condition, extra in [("normal", 0), ("hard", 40)]:
        if (case, condition) in done: continue
        img = sitk.ReadImage(path, sitk.sitkFloat32)
        truth = random_warp(img, seed=100 + i, extra_rot_deg=extra)
        tmp = tempfile.mkdtemp()
        fp, mp = os.path.join(tmp, "fixed.nii.gz"), os.path.join(tmp, "moving.nii.gz")
        fixed = sitk.Resample(img, img, truth, sitk.sitkLinear, 0.0, sitk.sitkFloat32)
        sitk.WriteImage(fixed, fp); sitk.WriteImage(img, mp)
        P = brain_points(fixed)
        T = np.array([truth.TransformPoint(p) for p in P])
        r, ants_map, fold = run_ants(fp, mp)
        A = ants_map(P)
        U = run_unigradicon(fp, mp, os.path.join(tmp, "u.hdf5"))(P)
        row = dict(case=case, group=group, condition=condition,
                   ants_p95_mm=np.percentile(np.linalg.norm(A - T, axis=1), 95),
                   ugi_p95_mm=np.percentile(np.linalg.norm(U - T, axis=1), 95),
                   disagree_mm=np.linalg.norm(A - U, axis=1).mean(),
                   ncc=ncc_after(fixed, r["warpedmovout"]), fold_pct=fold)
        for f in glob.glob(os.path.join(tmp, "*")): os.remove(f)
        pd.DataFrame([row]).to_csv(out_csv, mode="a", header=not os.path.exists(out_csv), index=False)
        print(case, condition, f"ANTs p95 {row['ants_p95_mm']:.1f} mm, NCC {row['ncc']:.3f}, disagree {row['disagree_mm']:.1f}")

30 bench cases
BraTS-SSA-00078-000 normal ANTs p95 1.4 mm, NCC 0.993, disagree 1.2
BraTS-SSA-00078-000 hard ANTs p95 1.7 mm, NCC 0.993, disagree 29.6
BraTS-SSA-00225-000 normal ANTs p95 1.3 mm, NCC 0.989, disagree 1.5
BraTS-SSA-00225-000 hard ANTs p95 1.3 mm, NCC 0.992, disagree 27.3
BraTS-SSA-00148-000 normal ANTs p95 1.2 mm, NCC 0.993, disagree 1.7
BraTS-SSA-00148-000 hard ANTs p95 1.2 mm, NCC 0.994, disagree 21.9
BraTS-SSA-00041-000 normal ANTs p95 1.0 mm, NCC 0.993, disagree 1.7
BraTS-SSA-00041-000 hard ANTs p95 1.1 mm, NCC 0.993, disagree 14.8
BraTS-SSA-00226-000 normal ANTs p95 0.8 mm, NCC 0.996, disagree 1.5
BraTS-SSA-00226-000 hard ANTs p95 44.5 mm, NCC 0.571, disagree 7.0
BraTS-SSA-00008-000 normal ANTs p95 1.3 mm, NCC 0.988, disagree 1.8
BraTS-SSA-00008-000 hard ANTs p95 1.4 mm, NCC 0.986, disagree 19.9
BraTS-SSA-00198-000 normal ANTs p95 1.2 mm, NCC 0.990, disagree 2.8
BraTS-SSA-00198-000 hard ANTs p95 1.2 mm, NCC 0.989, disagree 35.8
BraTS-SSA-00188-000 normal ANTs p95 1.0 

In [2]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
b = pd.read_csv("../results/bench_regcheck.csv")
b["ants_failed"] = b.ants_p95_mm > 2
print(pd.crosstab([b.group, b.condition], b.ants_failed))
for signal, higher_is_bad in [("ncc", False), ("disagree_mm", True), ("fold_pct", True)]:
    score = b[signal] if higher_is_bad else -b[signal]
    if b.ants_failed.nunique() == 2 and score.nunique() > 1:
        print(f"{signal}: AUC {roc_auc_score(b.ants_failed, score):.2f}")
    else:
        print(f"{signal}: cannot score (one class, or the signal never varies)")
ok, bad = b[~b.ants_failed], b[b.ants_failed]
print("NCC of good registrations: min", ok.ncc.min().round(3), "| failed registrations: max", bad.ncc.max().round(3))
print(b.groupby(["group", "condition"]).ncc.describe()[["min", "50%", "max"]].round(3))

ants_failed       False  True 
group  condition              
Africa hard          13      2
       normal        15      0
GLI    hard          12      3
       normal        15      0
ncc: AUC 1.00
disagree_mm: AUC 0.55
fold_pct: cannot score (one class, or the signal never varies)
NCC of good registrations: min 0.984 | failed registrations: max 0.759
                    min    50%    max
group  condition                     
Africa hard       0.571  0.991  0.995
       normal     0.988  0.992  0.996
GLI    hard       0.578  0.994  0.997
       normal     0.984  0.995  0.998
